# TradingAgents Workbench Debug Notebook

This notebook is split into 7 architecture parts so you can run, debug, and configure each layer independently.

In [1]:
# Global setup: make sure project root is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/workspaces/AlphaForgeFX')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

Project root: /workspaces/AlphaForgeFX


## Part 1: Bootstrap + Runtime Config

Load environment variables and apply a runtime configuration snapshot.

In [2]:
from tradingagents.workbench import load_environment, get_default_config_copy, apply_runtime_config

load_environment()
cfg = get_default_config_copy()
cfg['llm_provider'] = 'openai'
active_cfg = apply_runtime_config(cfg)

print('Active LLM provider:', active_cfg.get('llm_provider'))
print('Configured data vendors:', active_cfg.get('data_vendors'))

Active LLM provider: openai
Configured data vendors: {'core_stock_apis': 'yfinance', 'technical_indicators': 'yfinance', 'fundamental_data': 'yfinance', 'news_data': 'yfinance'}


## Part 2: LLM Layer

Create provider clients and inspect model wiring without running the graph.

In [3]:
from tradingagents.workbench import create_client

# This only constructs the client object; it does not run full trading flow
client = create_client(provider='openai', model='gpt-5.4-mini')
print('Client type:', type(client).__name__)

# Optional: uncomment to instantiate underlying llm object
# llm = client.get_llm()
# print(type(llm))

Client type: OpenAIClient


## Part 3: Data Layer

Inspect abstract data methods and available vendor backends.

In [4]:
from tradingagents.workbench import list_methods, list_method_vendors

methods = list_methods()
vendors = list_method_vendors()

print('Methods:', methods)
print('\nVendors per method:')
for method, backend_list in vendors.items():
    print(f'- {method}: {backend_list}')

Methods: ['get_balance_sheet', 'get_cashflow', 'get_fundamentals', 'get_global_news', 'get_income_statement', 'get_indicators', 'get_insider_transactions', 'get_news', 'get_stock_data']

Vendors per method:
- get_stock_data: ['alpha_vantage', 'yfinance']
- get_indicators: ['alpha_vantage', 'yfinance']
- get_fundamentals: ['alpha_vantage', 'yfinance']
- get_balance_sheet: ['alpha_vantage', 'yfinance']
- get_cashflow: ['alpha_vantage', 'yfinance']
- get_income_statement: ['alpha_vantage', 'yfinance']
- get_news: ['alpha_vantage', 'yfinance']
- get_global_news: ['alpha_vantage', 'yfinance']
- get_insider_transactions: ['alpha_vantage', 'yfinance']


In [5]:
from tradingagents.workbench import call

# Example routed call (requires provider/API availability)
# result = call('get_stock_data', 'AAPL', '2026-01-01', '2026-01-10')
# print(result)

print('Uncomment the call above to test a real routed data request.')

Uncomment the call above to test a real routed data request.


## Part 4: Tool Layer

Work directly with LangChain tool wrappers used by agents.

In [6]:
from tradingagents.workbench import list_tools, get_tool

tool_names = list_tools()
print('Tools:', tool_names)

stock_tool = get_tool('get_stock_data')
print('Tool object type:', type(stock_tool).__name__)

Tools: ['get_balance_sheet', 'get_cashflow', 'get_fundamentals', 'get_global_news', 'get_income_statement', 'get_indicators', 'get_insider_transactions', 'get_news', 'get_stock_data']
Tool object type: StructuredTool


In [7]:
from tradingagents.workbench import invoke_tool

# Example invocation (requires data provider availability)
# payload = {
#     'symbol': 'AAPL',
#     'start_date': '2026-01-01',
#     'end_date': '2026-01-10',
# }
# result = invoke_tool('get_stock_data', payload)
# print(result)

print('Uncomment the invocation above to run a real tool call.')

Uncomment the invocation above to run a real tool call.


## Part 5: Agent Layer

Inspect and access agent factory functions by role.

In [8]:
from tradingagents.workbench import list_factories, get_factory

factories = list_factories()
print('Factory count:', len(factories))
print(factories)

market_factory = get_factory('create_market_analyst')
print('Factory callable:', callable(market_factory))

Factory count: 15
['create_aggressive_debator', 'create_bear_researcher', 'create_bull_researcher', 'create_conservative_debator', 'create_executor', 'create_forex_trader', 'create_fundamentals_analyst', 'create_macro_analyst', 'create_market_analyst', 'create_neutral_debator', 'create_news_analyst', 'create_research_manager', 'create_risk_manager', 'create_social_media_analyst', 'create_trader']
Factory callable: True


## Part 6: Graph Layer

Build or run the orchestration graph independently from CLI.

In [9]:
from tradingagents.workbench import create_graph

graph = create_graph(debug=False, selected_analysts=['market', 'news', 'fundamentals'])
print('Graph type:', type(graph).__name__)

ValueError: OPENAI_API_KEY environment variable is not set. Please set it before initializing the TradingAgentsGraph.

In [ ]:
# Optional full run (requires valid API keys and configured providers)
# final_state, decision = graph.propagate('NVDA', '2026-01-15')
# print(decision)

print('Uncomment above when you are ready for a full graph run.')

## Part 7: Interface Layer

Access CLI app object for interface-level debugging/integration.

In [ ]:
from tradingagents.workbench import get_cli_app

app = get_cli_app()
print('CLI app type:', type(app).__name__)
app

## Quick Debug Checklist

1. If imports fail, run the first setup cell again.
2. If data calls fail, verify API keys and `data_vendors` in config.
3. If graph run fails, first test Parts 2-5 individually to isolate root cause.